# Week 12: Dataset Sources, APIs, and Google Drive Files

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST2412_Data_Security_Privacy_Ethics/blob/main/final_project/week12_dataset_sources_colab_examples.ipynb)

This notebook demonstrates beginner-friendly ways to use datasets for the final project.

You will see three patterns:

1. Load a class CSV directly from GitHub.
2. Load public data from easy web endpoints/APIs.
3. Mount Google Drive and load CSV files you downloaded yourself.

The point is not advanced coding. The point is to understand a dataset well enough to describe its columns, build a mini data dictionary, and document security, privacy, and ethics concerns.

## What You Need

This notebook uses only beginner Python tools:

- `pandas`: reads tables and helps summarize rows/columns
- `requests`: downloads data from a web URL or API endpoint
- `pathlib.Path`: helps work with file paths, especially Google Drive paths

If you run this in Colab, these tools are usually already available.

In [ ]:
# pandas is the main library we use for working with tables.
# Most CSV files become a pandas DataFrame.
# A DataFrame is like a spreadsheet: rows, columns, and values.
import pandas as pd

# requests lets us download data from web URLs and API endpoints.
import requests

# Path helps us write file paths in a cleaner way.
from pathlib import Path

# Show more columns when displaying DataFrames.
pd.set_option("display.max_columns", 50)

print("Setup complete.")

## Helper Function: Make a Starter Data Dictionary

A data dictionary explains what columns mean.

This helper does **not** finish the data dictionary for you. It creates a starter table so you can fill in the plain English meanings and concerns.

In [ ]:
def starter_data_dictionary(df, columns=None, max_columns=8):
    # Create a starter data dictionary for selected DataFrame columns.
    # df is the pandas DataFrame we want to document.
    # columns is an optional list of column names to include.
    # max_columns controls how many columns we use if no list is provided.

    # If the user does not choose columns, use the first few columns.
    if columns is None:
        columns = list(df.columns[:max_columns])

    rows = []

    for col in columns:
        # This is the full column as a pandas Series.
        series = df[col]

        # Drop blank values before choosing an example.
        non_blank = series.dropna()

        # Choose the first non-blank example value if one exists.
        example = non_blank.iloc[0] if len(non_blank) else ""

        rows.append({
            "column_name": col,
            "pandas_dtype": str(series.dtype),
            "example_value": example,
            "missing_count": int(series.isna().sum()),
            "unique_count": int(series.nunique(dropna=True)),
            "plain_english_meaning": "FILL THIS IN",
            "security_privacy_ethics_concern": "FILL THIS IN",
        })

    return pd.DataFrame(rows)

## Example 1: Load the Week 12 Class CSV from GitHub

This is the safest starter example because the dataset is part of the course repository.

Dataset: `week_12/data/week12_security_metrics.csv`

In [ ]:
# Raw GitHub URL for the Week 12 class CSV.
class_csv_url = "https://raw.githubusercontent.com/lolusername/CST2412_Data_Security_Privacy_Ethics/main/week_12/data/week12_security_metrics.csv"

# read_csv loads a CSV file into a DataFrame.
metrics = pd.read_csv(class_csv_url)

# shape tells us: (number of rows, number of columns).
print("Rows, columns:", metrics.shape)

# head shows the first 5 rows.
metrics.head()

In [ ]:
# Create a starter data dictionary for useful columns in the class dataset.
metrics_dictionary = starter_data_dictionary(
    metrics,
    columns=[
        "asset_id",
        "system_name",
        "business_unit",
        "asset_criticality",
        "internet_exposed",
        "sensitive_data",
        "failed_logins_24h",
        "highest_alert_severity",
    ],
)

metrics_dictionary

In [ ]:
# A simple beginner analyst task:
# create a priority score using columns from the class dataset.

# Make a copy so we do not accidentally change the original DataFrame.
metrics_scored = metrics.copy()

# map converts text labels into numbers.
# Here, high = 2, medium = 1, low = 0.
criticality_points = {"high": 2, "medium": 1, "low": 0}
severity_points = {"high": 2, "medium": 1, "low": 0}

def yes_no_points(value):
    # Return 1 for yes and 0 for everything else.
    return 1 if str(value).lower() == "yes" else 0

metrics_scored["priority_score"] = (
    metrics_scored["asset_criticality"].map(criticality_points)
    + metrics_scored["highest_alert_severity"].map(severity_points)
    + metrics_scored["internet_exposed"].apply(yes_no_points)
    + metrics_scored["sensitive_data"].apply(yes_no_points)
)

# Sort highest priority first.
metrics_scored.sort_values("priority_score", ascending=False).head(10)

## Example 2: Easy Security Dataset from CISA

CISA means the U.S. Cybersecurity and Infrastructure Security Agency.

The Known Exploited Vulnerabilities Catalog lists software/hardware vulnerabilities that have evidence of real-world exploitation.

This is directly relevant to security because it helps organizations prioritize what to patch or mitigate.

Official catalog page: https://www.cisa.gov/known-exploited-vulnerabilities-catalog

In [ ]:
# CISA provides a public JSON feed.
# JSON is a common web data format that Python can turn into dictionaries/lists.
kev_json_url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

response = requests.get(kev_json_url, timeout=30)
response.raise_for_status()  # stop if the request failed

kev_json = response.json()

# The vulnerability records are stored under the "vulnerabilities" key.
kev = pd.DataFrame(kev_json["vulnerabilities"])

print("Catalog title:", kev_json.get("title"))
print("Rows, columns:", kev.shape)
kev.head()

In [ ]:
# Build a data dictionary for CISA KEV columns.
kev_dictionary = starter_data_dictionary(
    kev,
    columns=[
        "cveID",
        "vendorProject",
        "product",
        "vulnerabilityName",
        "dateAdded",
        "requiredAction",
        "dueDate",
        "knownRansomwareCampaignUse",
    ],
)

kev_dictionary

In [ ]:
# Beginner analysis: which vendors appear most often in the catalog?

# value_counts counts how many times each value appears.
# reset_index turns the result back into a DataFrame.
top_vendors = (
    kev["vendorProject"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_vendors.columns = ["vendorProject", "count"]
top_vendors

### Security, Privacy, and Ethics Questions for CISA KEV

Possible notes for your lab or final project:

- Security: Which products or vendors appear repeatedly?
- Security: How could this help a beginner analyst prioritize patching?
- Ethics: What would be unfair about claiming a vendor is "bad" just because it appears often?
- Limitation: The catalog includes known exploited vulnerabilities, not every vulnerability.

## Example 3: Easy Public Data API from NYC Open Data

NYC Open Data uses Socrata-style API endpoints.

This example uses the Motor Vehicle Collisions dataset because it is public and easy to understand.

Dataset page: https://data.cityofnewyork.us/resource/h9gi-nx95

The code below downloads a small sample of rows, not the whole dataset.

In [ ]:
# This endpoint returns JSON rows from the NYC Open Data collisions dataset.
nyc_endpoint = "https://data.cityofnewyork.us/resource/h9gi-nx95.json"

# Parameters let us ask for only selected columns and only a limited number of rows.
params = {
    "$limit": 1000,
    "$select": "crash_date,crash_time,borough,zip_code,contributing_factor_vehicle_1,number_of_persons_injured,number_of_persons_killed",
}

response = requests.get(nyc_endpoint, params=params, timeout=30)
response.raise_for_status()

nyc_collisions = pd.DataFrame(response.json())

print("Rows, columns:", nyc_collisions.shape)
nyc_collisions.head()

In [ ]:
# Convert numeric-looking columns from text into numbers.
# errors='coerce' turns invalid/missing values into NaN instead of crashing.
for col in ["number_of_persons_injured", "number_of_persons_killed"]:
    nyc_collisions[col] = pd.to_numeric(nyc_collisions[col], errors="coerce")

# Build a starter data dictionary.
nyc_dictionary = starter_data_dictionary(nyc_collisions)
nyc_dictionary

In [ ]:
# Beginner analysis: top contributing factors in this sample.
factor_counts = (
    nyc_collisions["contributing_factor_vehicle_1"]
    .value_counts(dropna=False)
    .head(10)
    .reset_index()
)

factor_counts.columns = ["contributing_factor_vehicle_1", "count"]
factor_counts

### Security, Privacy, and Ethics Questions for NYC Open Data

Possible notes for your lab or final project:

- Privacy: Location plus time can reveal patterns, even when names are not present.
- Ethics: Neighborhood comparisons can be misleading without context.
- Limitation: A collision report is a recorded event, not a full explanation of why something happened.
- Data dictionary issue: Column names like `contributing_factor_vehicle_1` need plain English explanation.

## Example 4: Mount Google Drive and Load a Downloaded CSV

Use this section when you download a dataset yourself and place it in Google Drive.

### Recommended Google Drive Folder

Create this folder in your Google Drive:

`MyDrive/CST2412/week_12_datasets/`

### Recommended Files To Download

Pick **one** to start:

1. CISA KEV CSV
   - Download: https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv
   - Save to Google Drive as: `known_exploited_vulnerabilities.csv`

2. Week 12 class metrics CSV
   - Download: https://raw.githubusercontent.com/lolusername/CST2412_Data_Security_Privacy_Ethics/main/week_12/data/week12_security_metrics.csv
   - Save to Google Drive as: `week12_security_metrics.csv`

3. Your own NYC Open Data export
   - Go to https://data.cityofnewyork.us/
   - Choose a dataset
   - Use Export or Download CSV
   - Save to Google Drive as: `nyc_open_data_choice.csv`

4. Your own Kaggle CSV
   - Go to https://www.kaggle.com/datasets/
   - Choose a dataset related to security, privacy, technology, or data ethics
   - Download the CSV
   - Save to Google Drive as: `kaggle_security_dataset.csv`

Kaggle may require a free account. If that slows you down, use CISA KEV or the class CSV instead.

In [ ]:
# This cell works in Google Colab.
# It asks you to connect your Google Drive to the notebook.

try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except ModuleNotFoundError:
    print("This cell only mounts Drive in Google Colab. If you are not in Colab, skip it.")

In [ ]:
# Path to the recommended Google Drive folder.
drive_folder = Path("/content/drive/MyDrive/CST2412/week_12_datasets")

# Change this filename to match the CSV you put in Google Drive.
filename = "known_exploited_vulnerabilities.csv"

csv_path = drive_folder / filename

print("Looking for:", csv_path)

if csv_path.exists():
    drive_df = pd.read_csv(csv_path)
    print("Loaded from Drive. Rows, columns:", drive_df.shape)
    display(drive_df.head())
else:
    print("File not found yet.")
    print("Create this Google Drive folder:", drive_folder)
    print("Put your CSV file there, then run this cell again.")

In [ ]:
# If a Drive CSV loaded successfully, make a starter data dictionary for it.
if "drive_df" in globals():
    drive_dictionary = starter_data_dictionary(drive_df)
    display(drive_dictionary)
else:
    print("No Drive dataset loaded yet. Run the previous Drive loading cell first.")

## Your Turn: Choose One Dataset and Fill This In

Use either:

- the CISA KEV data loaded from the web
- the NYC Open Data sample loaded from the API
- a CSV you mounted from Google Drive
- the Week 12 class CSV

Answer these in your lab document:

1. Dataset source:
2. Dataset title:
3. Dataset URL or Drive filename:
4. What does one row represent?
5. Which 6-8 columns are most important?
6. What is one security concern?
7. What is one privacy concern?
8. What is one ethics concern or limitation?
9. How could this support your final project?

## Mini Data Dictionary Template

Copy this into your lab document and fill it in.

| Column name | Plain English meaning | Data type | Example value | Missing or unclear? | Security/privacy/ethics concern |
|---|---|---|---|---|---|
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |